# Title: Predicting Individual Medical Insurance Charges

## Contribution Statement

Individual project – all analysis, coding, and writing completed by the author.

## Problem Statement

The objective is to develop a regression model for medical insurance charges using demographic and health data. We aim to identify major predictors (e.g. smoking status, age, BMI) and ensure the final model meets classical OLS assumptions for valid inference and reliable prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11


#from google.colab import drive
#drive.mount('/content/drive')

#df = pd.read_csv('/content/drive/MyDrive/insurance.csv')

df = pd.read_csv('../insurance.csv')

print(f'Shape: {df.shape}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../insurance.csv'

## 4. Data Description and Exploratory Data Analysis

### 4.1 Variable Table

| Variable | Type | Description |
|---|---|---|
| `age` | Continuous | Age of the primary beneficiary (18–64) |
| `sex` | Categorical | Gender: male / female |
| `bmi` | Continuous | Body Mass Index (kg/m²) |
| `children` | Discrete | Number of dependents covered |
| `smoker` | Categorical | Smoking status: yes / no |
| `region` | Categorical | US residential region (4 levels) |
| `charges` | Continuous | Individual medical insurance cost (USD) — **response variable** |

### 4.2 Exploratory Data Analysis

In [ ]:
# ── Descriptive statistics ──────────────────────────────────────────────────
desc = df[['age', 'bmi', 'children', 'charges']].describe().round(2)
print('=== Descriptive Statistics (continuous variables) ===')
print(desc)
print()

print('=== Categorical variable counts ===')
for col in ['sex', 'smoker', 'region']:
    print(f'\n{col}:')
    print(df[col].value_counts())

In [ ]:
# ── Figure 1: Distributions of all variables ────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

# charges
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(df['charges'], bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(df['charges'].mean(), color='red', linestyle='--', linewidth=1.5,
            label=f"Mean = ${df['charges'].mean():,.0f}")
ax1.axvline(df['charges'].median(), color='orange', linestyle='--', linewidth=1.5,
            label=f"Median = ${df['charges'].median():,.0f}")
ax1.set_title('Distribution of Charges', fontweight='bold')
ax1.set_xlabel('Charges (USD)')
ax1.set_ylabel('Count')
ax1.legend(fontsize=9)

# age
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df['age'], bins=20, color='teal', edgecolor='white', linewidth=0.5)
ax2.set_title('Distribution of Age')
ax2.set_xlabel('Age')
ax2.set_ylabel('Count')

# bmi
ax3 = fig.add_subplot(gs[0, 3])
ax3.hist(df['bmi'], bins=30, color='mediumpurple', edgecolor='white', linewidth=0.5)
ax3.axvline(30, color='red', linestyle='--', linewidth=1.5, label='BMI = 30 (obese)')
ax3.set_title('Distribution of BMI')
ax3.set_xlabel('BMI')
ax3.set_ylabel('Count')
ax3.legend(fontsize=8)

# smoker
ax4 = fig.add_subplot(gs[1, 0])
smoker_counts = df['smoker'].value_counts()
ax4.bar(smoker_counts.index, smoker_counts.values,
        color=['salmon', 'lightgreen'], edgecolor='white')
for i, (k, v) in enumerate(smoker_counts.items()):
    ax4.text(i, v + 10, str(v), ha='center', fontsize=10)
ax4.set_title('Smoker')
ax4.set_ylabel('Count')

# sex
ax5 = fig.add_subplot(gs[1, 1])
sex_counts = df['sex'].value_counts()
ax5.bar(sex_counts.index, sex_counts.values,
        color=['cornflowerblue', 'lightcoral'], edgecolor='white')
for i, (k, v) in enumerate(sex_counts.items()):
    ax5.text(i, v + 10, str(v), ha='center', fontsize=10)
ax5.set_title('Sex')
ax5.set_ylabel('Count')

# children
ax6 = fig.add_subplot(gs[1, 2])
children_counts = df['children'].value_counts().sort_index()
ax6.bar(children_counts.index.astype(str), children_counts.values,
        color='goldenrod', edgecolor='white')
ax6.set_title('Number of Children')
ax6.set_xlabel('Children')
ax6.set_ylabel('Count')

# region
ax7 = fig.add_subplot(gs[1, 3])
region_counts = df['region'].value_counts()
ax7.bar(region_counts.index, region_counts.values,
        color='mediumseagreen', edgecolor='white')
ax7.set_title('Region')
ax7.set_xlabel('Region')
ax7.set_ylabel('Count')
ax7.tick_params(axis='x', labelsize=8)

fig.suptitle('Figure 1: Variable Distributions', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('fig1_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Charges vs. key predictors ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Figure 2: Charges vs. Each Predictor', fontsize=13, fontweight='bold')
plt.subplots_adjust(hspace=0.4, wspace=0.35)

colors = df['smoker'].map({'yes': 'tomato', 'no': 'steelblue'})

# charges vs age (coloured by smoker)
axes[0, 0].scatter(df['age'], df['charges'], c=colors, alpha=0.4, s=15)
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Charges (USD)')
axes[0, 0].set_title('Charges vs. Age\n(red = smoker, blue = non-smoker)')
for group, color in [('yes', 'darkred'), ('no', 'navy')]:
    sub = df[df['smoker'] == group]
    z = np.polyfit(sub['age'], sub['charges'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['age'].min(), sub['age'].max(), 100)
    axes[0, 0].plot(x_line, p(x_line), color=color, linewidth=1.5)

# charges vs bmi (coloured by smoker)
axes[0, 1].scatter(df['bmi'], df['charges'], c=colors, alpha=0.4, s=15)
axes[0, 1].set_xlabel('BMI')
axes[0, 1].set_ylabel('Charges (USD)')
axes[0, 1].set_title('Charges vs. BMI\n(red = smoker, blue = non-smoker)')
axes[0, 1].axvline(30, color='black', linestyle='--', linewidth=1, alpha=0.6, label='BMI=30')
axes[0, 1].legend(fontsize=8)

# charges vs children
axes[0, 2].boxplot(
    [df[df['children'] == k]['charges'].values for k in sorted(df['children'].unique())],
    labels=sorted(df['children'].unique())
)
axes[0, 2].set_xlabel('Number of Children')
axes[0, 2].set_ylabel('Charges (USD)')
axes[0, 2].set_title('Charges vs. Children')

# charges vs smoker
axes[1, 0].boxplot(
    [df[df['smoker'] == 'no']['charges'].values,
     df[df['smoker'] == 'yes']['charges'].values],
    labels=['Non-smoker', 'Smoker']
)
axes[1, 0].set_ylabel('Charges (USD)')
axes[1, 0].set_title('Charges vs. Smoker Status')

# charges vs sex
axes[1, 1].boxplot(
    [df[df['sex'] == 'female']['charges'].values,
     df[df['sex'] == 'male']['charges'].values],
    labels=['Female', 'Male']
)
axes[1, 1].set_ylabel('Charges (USD)')
axes[1, 1].set_title('Charges vs. Sex')

# charges vs region
regions = df['region'].unique()
axes[1, 2].boxplot(
    [df[df['region'] == r]['charges'].values for r in sorted(regions)],
    labels=sorted(regions)
)
axes[1, 2].set_ylabel('Charges (USD)')
axes[1, 2].set_title('Charges vs. Region')
axes[1, 2].tick_params(axis='x', labelsize=8)

plt.savefig('fig2_charges_vs_predictors.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: Correlation heatmap (numeric variables) ───────────────────────
df_corr = df.copy()
df_corr['smoker_yes'] = (df_corr['smoker'] == 'yes').astype(int)
df_corr['sex_male']   = (df_corr['sex'] == 'male').astype(int)

corr_cols = ['age', 'bmi', 'children', 'smoker_yes', 'sex_male', 'charges']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, ax=ax, linewidths=0.5,
    annot_kws={'size': 10}
)
ax.set_title('Figure 3: Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_correlation.png', bbox_inches='tight')
plt.show()

print('\nCorrelation with charges (sorted by absolute value):')
print(corr_matrix['charges'].drop('charges').sort_values(key=abs, ascending=False).round(3))

#### EDA Findings

**Charges distribution (response variable):** `charges` is strongly right-skewed (mean \$13,270 ≫ median \$9,382), with the distribution extending up to \$63,770. The histogram reveals a bimodal pattern — a large cluster of low-cost individuals and a smaller cluster of high-cost individuals. This shape suggests that a simple additive model will not capture the true data-generating process, and that a key binary driver (likely `smoker`) is separating the population into two distinct subgroups.

**Smoking is the dominant predictor:** The scatter plots of charges vs. `age` and vs. `bmi` (both colored by smoking status) reveal two nearly non-overlapping bands: smokers consistently incur far higher charges at every age and BMI level. The boxplot confirms that smokers' median charges are roughly 3–4× those of non-smokers. The correlation matrix quantifies this: `smoker_yes` has by far the highest correlation with charges (r = 0.79).

**Age has a positive linear effect within each group:** Both smokers and non-smokers show a clear upward trend in charges with age. The slopes appear similar across groups, suggesting an additive effect of `age`, though a quadratic term (age²) may capture further curvature.

**BMI shows a threshold effect — only among smokers:** Non-smokers show almost no sensitivity to BMI, while smokers above BMI ≈ 30 appear to cluster into a separate high-charge band. This motivates the `bmi × smoker` interaction and the `obese` binary feature (BMI ≥ 30).

**Sex and region are negligible:** The boxplots for sex and region show nearly identical charge distributions. The correlation matrix confirms this: `sex_male` (r = 0.06) is essentially uncorrelated with charges, and region adds little predictive signal.

**Correlation summary:** `smoker_yes` (r = 0.79) >> `age` (r = 0.30) > `bmi` (r = 0.20) > `children` (r = 0.07) > `sex_male` (r = 0.06). The model should prioritize smoking status and its interactions.

### 4.3 Data Engineering

In [ ]:
# ── Create the obese feature ─────────────────────────────────────────────────
# obese: 1 if BMI >= 30 (WHO clinical threshold for obesity), 0 otherwise
df['obese'] = (df['bmi'] >= 30).astype(int)

print('=== obese feature ===')
print(f"  BMI >= 30 (obese):  {df['obese'].sum():4d}  ({df['obese'].mean()*100:.1f}%)")
print(f"  BMI <  30:          {(df['obese']==0).sum():4d}  ({(df['obese']==0).mean()*100:.1f}%)")
print()

# Preview
df[['age', 'bmi', 'obese', 'smoker', 'charges']].head(8)

In [ ]:
# ── Figure 6: Charges by obese × smoker group ───────────────────────────────
smoker_bin = (df['smoker'] == 'yes').astype(int)

groups = {
    'Non-obese\nNon-smoker': df[(df['obese']==0) & (smoker_bin==0)]['charges'],
    'Non-obese\nSmoker':     df[(df['obese']==0) & (smoker_bin==1)]['charges'],
    'Obese\nNon-smoker':     df[(df['obese']==1) & (smoker_bin==0)]['charges'],
    'Obese\nSmoker':         df[(df['obese']==1) & (smoker_bin==1)]['charges'],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 6: The obese Feature — Distribution and Effect on Charges',
             fontsize=12, fontweight='bold')

# (a) obese counts
obese_counts = df['obese'].value_counts().sort_index()
axes[0].bar(['Non-obese (BMI < 30)', 'Obese (BMI ≥ 30)'],
            obese_counts.values,
            color=['cornflowerblue', 'tomato'], edgecolor='white')
for i, v in enumerate(obese_counts.values):
    axes[0].text(i, v + 10, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_ylabel('Count')
axes[0].set_title('(a) Obese vs. Non-obese')

# (b) Charges by obese × smoker (4 groups)
bp = axes[1].boxplot(groups.values(), labels=groups.keys(), patch_artist=True)
colors_box = ['lightblue', 'lightsalmon', 'cornflowerblue', 'tomato']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
axes[1].set_ylabel('Charges (USD)')
axes[1].set_title('(b) Charges by Obese × Smoker Group')
axes[1].tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig('fig6_obese_feature.png', bbox_inches='tight')
plt.show()

print('Mean charges by group:')
for name, grp in groups.items():
    print(f'  {name.replace(chr(10), " "):30s}  n={len(grp):3d}  mean=${grp.mean():,.0f}')

## 5. Baseline Model and Preliminary Diagnostics

### 5.1 Preview of Assumption Violations

Before building the full model, we fit the simplest possible baseline — OLS with only the three continuous predictors (`age`, `bmi`, `children`) — and formally test all classical OLS assumptions. This motivates every modeling decision in Section 6.

In [ ]:
# ── Baseline model M1: charges ~ age + bmi + children ───────────────────────
X_m1 = sm.add_constant(df[['age', 'bmi', 'children']])
m1 = sm.OLS(df['charges'], X_m1).fit()
print(m1.summary())

In [ ]:
# ── Figure 4: Diagnostic plots for M1 ───────────────────────────────────────
fitted_m1    = m1.fittedvalues
resid_m1     = m1.resid
std_resid_m1 = resid_m1 / resid_m1.std()
influence_m1 = m1.get_influence()
leverage_m1  = influence_m1.hat_matrix_diag

order = fitted_m1.argsort()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Figure 4: Baseline Model (M1) — Diagnostic Plots', fontsize=12, fontweight='bold')

# (a) Residuals vs. Fitted
axes[0].scatter(fitted_m1, resid_m1, alpha=0.3, s=15, color='steelblue')
axes[0].axhline(0, color='grey', linewidth=1)
smooth = lowess(resid_m1.values[order], fitted_m1.values[order], frac=0.3)
axes[0].plot(smooth[:, 0], smooth[:, 1], color='red', linewidth=1.5)
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('(a) Residuals vs. Fitted')

# (b) Normal Q-Q
sm.qqplot(resid_m1, line='s', alpha=0.4, ax=axes[1], markersize=3)
axes[1].set_title('(b) Normal Q-Q')

# (c) Scale-Location
sqrt_std_resid = np.sqrt(np.abs(std_resid_m1))
axes[2].scatter(fitted_m1, sqrt_std_resid, alpha=0.3, s=15, color='steelblue')
smooth2 = lowess(sqrt_std_resid.values[order], fitted_m1.values[order], frac=0.3)
axes[2].plot(smooth2[:, 0], smooth2[:, 1], color='red', linewidth=1.5)
axes[2].set_xlabel('Fitted Values')
axes[2].set_ylabel('√|Standardized Residuals|')
axes[2].set_title('(c) Scale-Location')

plt.tight_layout()
plt.savefig('fig4_m1_diagnostics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Formal violation tests for M1 ───────────────────────────────────────────

# 1. Breusch-Pagan test (heteroscedasticity)
bp_stat, bp_pval, _, _ = het_breuschpagan(resid_m1, X_m1)

# 2. Durbin-Watson test (autocorrelation)
dw_stat = durbin_watson(resid_m1)

# 3. Shapiro-Wilk test (normality)
sw_stat, sw_pval = stats.shapiro(resid_m1.sample(min(len(resid_m1), 5000), random_state=42))

# 4. RESET test (nonlinearity) — add fitted² and fitted³ as auxiliary regressors
fitted2 = fitted_m1 ** 2
fitted3 = fitted_m1 ** 3
X_reset = np.column_stack([X_m1, fitted2, fitted3])
m1_reset = sm.OLS(df['charges'], X_reset).fit()
reset_fstat = ((m1_reset.ssr - m1.ssr) / 2) / (m1.ssr / m1.df_resid)
reset_pval  = 1 - stats.f.cdf(reset_fstat, 2, m1.df_resid)

alpha = 0.05
print('=' * 55)
print('  Formal Violation Tests — Baseline Model M1')
print('=' * 55)
print(f'  Breusch-Pagan test   BP = {bp_stat:.3f},  p = {bp_pval:.2e}')
print(f'  Durbin-Watson stat   DW = {dw_stat:.4f}')
print(f'  Shapiro-Wilk test    W  = {sw_stat:.4f},  p = {sw_pval:.2e}')
print(f'  RESET test           F  = {reset_fstat:.3f},  p = {reset_pval:.2e}')
print('=' * 55)
print(f'\n  Heteroscedasticity: {"VIOLATED" if bp_pval < alpha else "OK"} (BP p = {bp_pval:.2e})')
print(f'  Autocorrelation:    DW = {dw_stat:.4f} (near 2 = OK)')
print(f'  Normality:          {"VIOLATED" if sw_pval < alpha else "OK"} (SW p = {sw_pval:.2e})')
print(f'  Nonlinearity:       {"VIOLATED" if reset_pval < alpha else "OK"} (RESET p = {reset_pval:.2e})')

In [ ]:
# ── VIF for M1 (multicollinearity check) ────────────────────────────────────
vif_data = pd.DataFrame({
    'Variable': ['age', 'bmi', 'children'],
    'VIF': [
        variance_inflation_factor(X_m1.values, i + 1)  # skip constant (col 0)
        for i in range(3)
    ]
}).round(3)

print('VIF Table — Baseline Model M1')
print(vif_data.to_string(index=False))

In [ ]:
# ── Figure 5: Cook's Distance ────────────────────────────────────────────────
cooks_d   = influence_m1.cooks_distance[0]
threshold = 4 / len(df)   # rule-of-thumb: 4/n
n_influ   = (cooks_d > threshold).sum()

fig, ax = plt.subplots(figsize=(10, 3))
ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',',
        linefmt='steelblue', basefmt='grey')
ax.axhline(threshold, color='red', linestyle='--', linewidth=1.2,
           label=f'Threshold = 4/n = {threshold:.4f}')
ax.set_xlabel('Observation Index')
ax.set_ylabel("Cook's Distance")
ax.set_title("Figure 5: Cook's Distance — Influential Points (M1)", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig5_cooks_distance.png', bbox_inches='tight')
plt.show()

print(f"Influential points (Cook's D > 4/n): {n_influ} ({n_influ/len(df)*100:.1f}% of observations)")

#### Violation Summary

The baseline model M1 (`charges ~ age + bmi + children`) violates multiple classical OLS assumptions:

| Assumption | Test | Verdict | Implication |
|---|---|---|---|
| **Homoscedasticity** | Breusch-Pagan (BP) | **VIOLATED** (p ≪ 0.05) | Residual variance grows with fitted values; `smoker` is the likely hidden driver |
| **Normality of residuals** | Shapiro-Wilk (SW) | **VIOLATED** (p ≪ 0.05) | Heavy right tail due to two distinct subpopulations (smokers vs. non-smokers) |
| **Independence** | Durbin-Watson (DW) | **OK** (DW ≈ 2) | No evidence of autocorrelation; expected for cross-sectional survey data |
| **Linearity** | RESET test | OK | No nonlinearity when only continuous predictors are used |
| **Multicollinearity** | VIF | **OK** (all VIF < 5) | No concern among `age`, `bmi`, `children` |

The residual vs. fitted plot (Figure 4a) shows a pronounced **two-band pattern** — the unmistakable signature of a missing binary predictor separating the data into two subpopulations. Including `smoker` (Section 6) will directly address both the heteroscedasticity and the non-normality. The diagnostic results collectively motivate adding interaction terms (especially `bmi × smoker`) and polynomial terms (`age²`) in the models that follow.

## 6. Candidate Regression Models

Multiple regression models are tested to address the issues identified in the baseline diagnostics (Section 5). The baseline model M1 violated homoscedasticity and normality, largely due to the omitted `smoker` variable. The following candidate models (M2–M6) incorporate different modeling strategies to improve fit and satisfy OLS assumptions.

### 6.1 Model Selection Strategy

We now fit six candidate regression models to compare their ability to predict insurance charges while satisfying OLS assumptions. Each model represents a different modeling philosophy:

- **M1 (Baseline)**: Continuous predictors only → violated assumptions, motivates inclusion of `smoker`
- **M2 (Full)**: All predictors, no interactions → captures smoking effect but misses synergies
- **M3 (Polynomial)**: Adds `age²` to capture age curvature
- **M4 (Interaction)**: Adds crucial `bmi × smoker` interaction
- **M5 (Log-transformed)**: Addresses skewness via log(charges) transformation
- **M6 (Obese interaction)**: Uses binary obese (BMI ≥ 30) with obese×smoker interaction → best fit

For each model:
- Fit using statsmodels OLS
- Apply all diagnostic methods from Section 5
- Compute selection criteria (R², AIC, BIC, RMSE)
- Compare diagnostic plots and tests


#### Candidate Models M2–M6

##### Model M2 — Full Model
**charges ~ age + bmi + children + smoker + sex + region**

Includes all available predictors. Captures the strong smoking effect and regional/sex differences.

##### Model M3 — Polynomial Model
**charges ~ age + age² + bmi + children + smoker**

Adds a quadratic age term to capture nonlinear age effects (e.g., charges may increase more rapidly at older ages).

##### Model M4 — Interaction Model
**charges ~ age + bmi + smoker + bmi×smoker + children**

Smoking may amplify the effect of BMI: obese smokers may face disproportionately higher charges than obese non-smokers.

##### Model M5 — Log-Transformed Model
**log(charges) ~ age + bmi + smoker + children**

The log transformation can reduce right skewness and heteroscedasticity in the response.

##### Model M6 — Interaction Model with obese feature
**charges ~ age + obese + smoker + obese×smoker + children**

##### Model M7 — Bidirectional Stepwise Feature Selection Model

Uses the binary obese indicator (BMI ≥ 30) instead of continuous BMI. Captures the obese×smoker interaction to test whether obesity amplifies charges among smokers. BIDIRECTIONAL STEPWISE FEATURE SELECTION (AIC-based)
  Candidate features (10): age, age², obese, children, smoker,
    sex, region (3 dummies), obese×smoker
  Note: bmi (continuous) excluded; only obese (BMI ≥ 30) used

In [ ]:

# ── Prepare data: Create dummy variables ────────────────────────────────────
df['smoker_yes'] = (df['smoker'] == 'yes').astype(int)
df['sex_male']   = (df['sex'] == 'male').astype(int)

# Create region dummies (drop one for reference)
region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=True)
df = df.join(region_dummies)

# Create age squared
df['age_sq'] = df['age'] ** 2

# Create log of charges
df['log_charges'] = np.log(df['charges'])

print('Data preparation complete.')
print(f"Total observations: {len(df)}")
df[['age', 'bmi', 'smoker_yes', 'age_sq', 'charges', 'log_charges']].head()


In [ ]:
print(df.columns.tolist())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL 2: Full model (all predictors, no interactions or polynomials)
# charges ~ age + bmi + children + smoker + sex + region
# ──────────────────────────────────────────────────────────────────────────────

m2_formula = 'charges ~ age + bmi + children + smoker_yes + sex_male + region_southwest + region_northwest + region_southeast'
m2 = smf.ols(m2_formula, data=df).fit()

print('\n' + '='*70)
print('  MODEL 2: Full Model (age + bmi + children + smoker + sex + region)')
print('='*70)
print(m2.summary())


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL 3: Polynomial model (adds age²)
# charges ~ age + age² + bmi + children + smoker
# ──────────────────────────────────────────────────────────────────────────────

m3_formula = 'charges ~ age + age_sq + bmi + children + smoker_yes'
m3 = smf.ols(m3_formula, data=df).fit()

print('\n' + '='*70)
print('  MODEL 3: Polynomial Model (age + age² + bmi + children + smoker)')
print('='*70)
print(m3.summary())


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL 4: Interaction model (includes bmi × smoker interaction)
# charges ~ age + bmi + smoker + bmi*smoker + children
# ──────────────────────────────────────────────────────────────────────────────

# Create interaction term manually
df['bmi_smoker_int'] = df['bmi'] * df['smoker_yes']

m4_formula = 'charges ~ age + bmi + smoker_yes + bmi_smoker_int + children'
m4 = smf.ols(m4_formula, data=df).fit()

print('\n' + '='*70)
print('  MODEL 4: Interaction Model (age + bmi + smoker + bmi×smoker + children)')
print('='*70)
print(m4.summary())


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL 5: Log-transformed model (addresses right skewness)
# log(charges) ~ age + bmi + smoker + children
# ──────────────────────────────────────────────────────────────────────────────

m5_formula = 'log_charges ~ age + bmi + smoker_yes + children'
m5 = smf.ols(m5_formula, data=df).fit()

print('\n' + '='*70)
print('  MODEL 5: Log-Transformed Model (log(charges) ~ age + bmi + smoker + children)')
print('='*70)
print(m5.summary())


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL 6: Interaction model with obese feature
# charges ~ age + obese + smoker + obese×smoker + children
# ──────────────────────────────────────────────────────────────────────────────

# Create obese×smoker interaction (obese is from Data Engineering, Section 4.3)
df['obese_smoker_int'] = df['obese'] * df['smoker_yes']

m6_formula = 'charges ~ age + obese + smoker_yes + obese_smoker_int + children'
m6 = smf.ols(m6_formula, data=df).fit()

print('\n' + '='*70)
print('  MODEL 6: Interaction Model with obese (charges ~ age + obese + smoker + obese×smoker + children)')
print('='*70)
print(m6.summary())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# M7: BIDIRECTIONAL STEPWISE FEATURE SELECTION (AIC-based)
# At each step: consider BOTH adding an excluded feature AND removing an included
# feature; choose whichever lowers AIC most. Stop when no improvement is possible.
#
# Feature set: bmi is excluded — only the binary obese indicator (BMI ≥ 30) is
# used as the BMI-related predictor, together with the obese×smoker interaction.
# ──────────────────────────────────────────────────────────────────────────────

# Ensure interaction term exists
df['obese_smoker_int'] = df['obese'] * df['smoker_yes']

# Candidate feature set (10 predictors — no bmi, no bmi_smoker_int)
all_features = [
    'age',
    'obese',
    'children',
    'smoker_yes',
    'sex_male',
    'region_northwest', 'region_southeast', 'region_southwest',
    'obese_smoker_int'
]

def stepwise_select(data, response, all_features, direction='both', start='full'):
    """
    Bidirectional stepwise selection using AIC criterion.
    At each step both addition and removal are evaluated; the action that
    produces the largest AIC reduction is applied.  Stops when no single
    addition or removal can further reduce AIC.

    Parameters
    ----------
    direction : 'forward', 'backward', or 'both'
    start     : 'full'  — begin with all candidate features (backward-style start)
                'empty' — begin with no features (forward-style start)
    """
    included = list(all_features) if start == 'full' else []

    def get_aic(features):
        formula = response + (" ~ " + " + ".join(features) if features else " ~ 1")
        return smf.ols(formula, data=data).fit().aic

    current_aic = get_aic(included)
    step = 0
    print(f"Start ({start}): {len(included)} features, AIC = {current_aic:.2f}")
    print(f"Initial features: {included}\n")

    while True:
        best_aic    = current_aic
        best_action = None
        best_feat   = None

        # --- Forward step: try adding each excluded feature ---
        if direction in ('forward', 'both'):
            for f in [x for x in all_features if x not in included]:
                aic = get_aic(included + [f])
                if aic < best_aic:
                    best_aic, best_action, best_feat = aic, 'add', f

        # --- Backward step: try removing each included feature ---
        if direction in ('backward', 'both') and included:
            for f in included:
                remaining = [x for x in included if x != f]
                aic = get_aic(remaining)
                if aic < best_aic:
                    best_aic, best_action, best_feat = aic, 'remove', f

        if best_action is None:
            print(f"\nConverged after {step} step(s) — no further AIC improvement.")
            break

        step += 1
        if best_action == 'add':
            included.append(best_feat)
            print(f"Step {step:2d}: ADD    '{best_feat}'  → AIC = {best_aic:.2f}")
        else:
            included.remove(best_feat)
            print(f"Step {step:2d}: REMOVE '{best_feat}'  → AIC = {best_aic:.2f}")
        current_aic = best_aic

    formula = response + " ~ " + " + ".join(included)
    return included, smf.ols(formula, data=data).fit()


print("=" * 70)
print("  BIDIRECTIONAL STEPWISE FEATURE SELECTION (AIC-based)")
print("  Candidate features (10): age, age², obese, children, smoker,")
print("    sex, region (3 dummies), obese×smoker")
print("  Note: bmi (continuous) excluded; only obese (BMI ≥ 30) used")
print("=" * 70 + "\n")

selected_features, m_bwd = stepwise_select(
    df, "charges", all_features, direction='both', start='full'
)

print(f"\nFinal selected features ({len(selected_features)}): {selected_features}")
print("\n" + "=" * 70)
print(" M7: BIDIRECTIONAL STEPWISE")
print("=" * 70)
print(m_bwd.summary())

### 6.2 Model Comparison

Summary of the six candidate models:

| Model | Formula | Key Feature | Response |
|-------|---------|-------------|----------|
| M1 | charges ~ age + bmi + children | Continuous only | charges |
| M2 | charges ~ ... + smoker + sex + region | All predictors | charges |
| M3 | charges ~ age + age² + bmi + ... + smoker | Polynomial (age²) | charges |
| M4 | charges ~ age + bmi + smoker + **bmi×smoker** + children | Interaction (bmi×smoker) | charges |
| M5 | log(charges) ~ age + bmi + smoker + children | Log transformation | log(charges) |
| M6 | charges ~ age + obese + smoker + **obese×smoker** + children | Obese interaction | charges |


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Diagnostics for Each Candidate Model (M2, M3, M4, M5)
# Apply: residual plots, Breusch-Pagan, VIF, Cook's distance, Durbin-Watson
# ──────────────────────────────────────────────────────────────────────────────

def run_model_diagnostics(model, model_name, X_names, fig_prefix=''):
    """Run full diagnostic suite for a fitted OLS model."""
    resid = model.resid
    fitted = model.fittedvalues
    X = model.model.exog

    # 1. Breusch-Pagan test
    bp_stat, bp_pval, _, _ = het_breuschpagan(resid, X)
    # 2. Durbin-Watson
    dw_stat = durbin_watson(resid)
    # 3. VIF (skip constant column 0)
    n_vars = X.shape[1] - 1
    vifs = [variance_inflation_factor(X, i + 1) for i in range(n_vars)]
    # 4. Cook's distance
    influence = model.get_influence()
    cooks_d = influence.cooks_distance[0]
    threshold = 4 / len(resid)
    n_influ = (cooks_d > threshold).sum()

    print(f'\n{"="*70}')
    print(f'  DIAGNOSTICS: {model_name}')
    print('='*70)
    print(f'  Breusch-Pagan:  BP = {bp_stat:.3f},  p = {bp_pval:.2e}  {"(VIOLATED)" if bp_pval < 0.05 else "(OK)"}')
    print(f'  Durbin-Watson:  DW = {dw_stat:.4f}  (near 2 = OK)')
    print(f"  Cook's D > 4/n: {n_influ} influential points ({n_influ/len(resid)*100:.1f}%)")
    print('  VIF:')
    for name, vif in zip(X_names, vifs):
        print(f'    {name:25s} {vif:.3f}')

    # Residual diagnostic plots (2x2)
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    # (a) Residuals vs Fitted
    axes[0,0].scatter(fitted, resid, alpha=0.5, s=20)
    axes[0,0].axhline(0, color='red', linestyle='--')
    axes[0,0].set_xlabel('Fitted values')
    axes[0,0].set_ylabel('Residuals')
    axes[0,0].set_title(f'(a) Residuals vs Fitted — {model_name}')
    # (b) Q-Q plot
    stats.probplot(resid, dist="norm", plot=axes[0,1])
    axes[0,1].set_title(f'(b) Q-Q Plot — {model_name}')
    # (c) Scale-Location
    axes[1,0].scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.5, s=20)
    axes[1,0].set_xlabel('Fitted values')
    axes[1,0].set_ylabel('√|Residuals|')
    axes[1,0].set_title(f'(c) Scale-Location — {model_name}')
    # (d) Cook's distance
    axes[1,1].stem(range(len(cooks_d)), cooks_d, markerfmt=',', linefmt='steelblue', basefmt='grey')
    axes[1,1].axhline(threshold, color='red', linestyle='--', label=f'4/n = {threshold:.4f}')
    axes[1,1].set_xlabel('Observation Index')
    axes[1,1].set_ylabel("Cook's Distance")
    axes[1,1].set_title(f"(d) Cook's Distance — {model_name}")
    axes[1,1].legend()
    plt.tight_layout()
    if fig_prefix:
        plt.savefig(f'{fig_prefix}_diagnostics.png', bbox_inches='tight')
    plt.show()

# Run diagnostics for M2
X_m2_names = ['age','bmi','children','smoker_yes','sex_male','region_northeast','region_northwest','region_southeast']
run_model_diagnostics(m2, 'M2 Full Model', X_m2_names, 'm2')

In [ ]:
# Diagnostics for M3, M4, M5, M6, M7
X_m3_names = ['age', 'age_sq', 'bmi', 'children', 'smoker_yes']
run_model_diagnostics(m3, 'M3 Polynomial Model', X_m3_names, 'm3')

X_m4_names = ['age', 'bmi', 'smoker_yes', 'bmi_smoker_int', 'children']
run_model_diagnostics(m4, 'M4 Interaction Model', X_m4_names, 'm4')

# M5: response is log(charges); diagnostics in log space
X_m5_names = ['age', 'bmi', 'smoker_yes', 'children']
run_model_diagnostics(m5, 'M5 Log-Transformed Model', X_m5_names, 'm5')

# M6: obese interaction model
X_m6_names = ['age', 'obese', 'smoker_yes', 'obese_smoker_int', 'children']
run_model_diagnostics(m6, 'M6 Obese Interaction Model', X_m6_names, 'm6')

# M7: bidirectional stepwise model (features determined by AIC selection)
X_mbwd_names = [n for n in m_bwd.model.exog_names if n != 'Intercept']
run_model_diagnostics(m_bwd, 'M7 Stepwise Model', X_mbwd_names, 'm_bwd')

#### Comparison Metrics

We compare all models using standard fit and information criteria. **R²** measures the proportion of variance explained; **Adjusted R²** penalizes extra predictors. **RMSE** (root mean squared error) quantifies prediction error in the original charge scale. **AIC** and **BIC** balance fit and parsimony—lower values indicate better models. *Note: M5 models log(charges), so its R², AIC, and BIC are not directly comparable to charge-scale models (M1–M4, M6); only RMSE is comparable for M5 since it is back-transformed to the charge scale.*

In [ ]:
# ── Build comparison table: R², Adj R², RMSE, AIC, BIC ──────────────────────
# For M5, RMSE is computed in original charge scale: exp(pred_log) vs actual charges
pred_m1   = m1.fittedvalues
pred_m2   = m2.fittedvalues
pred_m3   = m3.fittedvalues
pred_m4   = m4.fittedvalues
pred_m5   = np.exp(m5.fittedvalues)   # back-transform to charges
pred_m6   = m6.fittedvalues
pred_mbwd = m_bwd.fittedvalues

rmse = lambda y, p: np.sqrt(np.mean((y - p)**2))
y = df['charges'].values

comparison = pd.DataFrame({
    'Model': ['M1 Baseline', 'M2 Full', 'M3 Polynomial',
              'M4 Interaction', 'M5 Log', 'M6 Obese', 'M_bwd Stepwise'],
    'R²':     [m1.rsquared,     m2.rsquared,     m3.rsquared,
               m4.rsquared,     m5.rsquared,     m6.rsquared,     m_bwd.rsquared],
    'Adj R²': [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj,
               m4.rsquared_adj, m5.rsquared_adj, m6.rsquared_adj, m_bwd.rsquared_adj],
    'RMSE':   [rmse(y, pred_m1), rmse(y, pred_m2), rmse(y, pred_m3),
               rmse(y, pred_m4), rmse(y, pred_m5), rmse(y, pred_m6), rmse(y, pred_mbwd)],
    'AIC':    [m1.aic, m2.aic, m3.aic, m4.aic, m5.aic, m6.aic, m_bwd.aic],
    'BIC':    [m1.bic, m2.bic, m3.bic, m4.bic, m5.bic, m6.bic, m_bwd.bic]
}).round(4)

print('Model Comparison Table')
print('='*80)
print(comparison.to_string(index=False))
print('='*80)
print('\nLower RMSE, AIC, BIC = better. Higher R², Adj R² = better.')
print(f'\nM_bwd selected features ({len(selected_features)}): {selected_features}')

### Predicted vs Actual Plots

Predicted vs. actual plots show how well each model fits the data. Points close to the **45-degree reference line (y = x)** indicate accurate predictions. Systematic deviations from the line suggest bias or misspecification.

In [ ]:
# ── Predicted vs Actual plots — all candidate models (2×3 grid) ──────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
models_info = [
    (m2,    pred_m2,   'M2 Full'),
    (m3,    pred_m3,   'M3 Polynomial'),
    (m4,    pred_m4,   'M4 Interaction'),
    (m5,    pred_m5,   'M5 Log-Transformed'),
    (m6,    pred_m6,   'M6 Obese'),
    (m_bwd, pred_mbwd, 'M_bwd Stepwise'),
]
for ax, (model, pred, name) in zip(axes.flat, models_info):
    ax.scatter(y, pred, alpha=0.5, s=20)
    lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='y = x (perfect fit)')
    ax.set_xlabel('Actual charges ($)')
    ax.set_ylabel('Predicted charges ($)')
    ax.set_title(f'Predicted vs Actual — {name}')
    ax.legend()
    ax.set_aspect('equal')
plt.suptitle('Predicted vs Actual Charges — All Candidate Models', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', bbox_inches='tight')
plt.show()

## 7. Conclusions

### Best Model

Based on the model comparison table and diagnostic results, **Model M6 (Obese Interaction Model)** is the **final recommended model** for predicting medical insurance charges. Among the charge-scale models (M1–M4, M6), M6 achieves the highest R² (0.86) and Adjusted R², the lowest RMSE (~4,511), and the lowest AIC and BIC. The use of the binary **obese** indicator (BMI ≥ 30) with the **obese × smoker** interaction captures the synergistic effect whereby smoking amplifies the impact of obesity on charges—obese smokers face disproportionately higher charges than obese non-smokers. M6 outperforms M4 (continuous BMI interaction) by treating obesity as a clinical threshold rather than a continuous scale, which better matches the observed charge structure in the data.

### Diagnostic Assumptions

Compared with the baseline M1, the candidate models show improved diagnostic behavior:
- **Homoscedasticity**: Including `smoker` reduces the two-band residual pattern. M4 and M6 show relatively better residual spread among charge-scale models, though some heteroscedasticity may remain.
- **Normality**: Residuals improve with the inclusion of smoking status. The log-transformed model (M5) addresses skewness in log space, but its R² and AIC/BIC are not comparable to charge-scale models.
- **Multicollinearity**: VIF values remain acceptable (< 5) for all models.
- **Influential points**: Cook's distance identifies a modest number of influential observations; these warrant review but do not invalidate the model.

### Key Findings

1. **Smoking has a strong effect**: Smokers incur substantially higher charges than non-smokers across all models. This was the primary driver of baseline violations.
2. **Obese × smoking interaction**: The interaction term in M6 is significant and substantively important. Obese smokers face disproportionately higher charges than obese non-smokers. Using the binary obese indicator (BMI ≥ 30) outperforms the continuous BMI interaction in M4.
3. **Nonlinear age effects**: The polynomial term in M3 (age²) can capture curvature in the age–charges relationship, though M6's obese interaction provides greater improvement in fit.
4. **Log transformation**: M5 addresses skewness and can improve residual behavior in log space, but its R² is not comparable to charge-scale models. M6 offers the best predictive performance in the original charge scale.

### Final Recommended Model

**M6: charges ~ age + obese + smoker + obese×smoker + children**

This model balances interpretability, predictive accuracy, and reasonable compliance with OLS assumptions. It is suitable for inference on the effects of smoking, obesity (BMI ≥ 30), and their interaction, as well as for prediction of individual medical insurance charges.

## Appendix

Additional diagnostic plots and supplementary tables are provided below for reference.

In [ ]:
# ── Appendix: Breusch-Pagan and Durbin-Watson summary for all models ─────────
all_models = [
    ('M1',          m1),
    ('M2',          m2),
    ('M3',          m3),
    ('M4',          m4),
    ('M5',          m5),
    ('M6',          m6),
    ('M7',       m_bwd),
]

diag_rows = []
for label, mdl in all_models:
    bp_stat, bp_pval, _, _ = het_breuschpagan(mdl.resid, mdl.model.exog)
    dw = durbin_watson(mdl.resid)
    diag_rows.append({
        'Model':      label,
        'BP_stat':    bp_stat,
        'BP_pval':    bp_pval,
        'DW':         dw,
        'BP_verdict': 'VIOLATED' if bp_pval < 0.05 else 'OK',
    })

diag_summary = pd.DataFrame(diag_rows)
print('Appendix Table: Breusch-Pagan and Durbin-Watson by Model')
print(diag_summary.to_string(index=False))

#### Final Model Summary (M6)

The table below highlights the key coefficients of the selected obese interaction model (M6). It summarizes the estimated effect of each predictor on medical insurance charges, along with standard errors, test statistics, and confidence intervals. This presentation helps interpret the practical meaning of each term and supports inference about which factors are most strongly associated with charges.

In [ ]:
# ── M6 Coefficient Summary Table ─────────────────────────────────────────────
# Extract from fitted M6 model (charges ~ age + obese + smoker + obese*smoker + children)
label_map = {
    'Intercept': 'Intercept',
    'age': 'age',
    'obese': 'obese',
    'smoker_yes': 'smoker',
    'obese_smoker_int': 'obese:smoker',
    'children': 'children'
}
interpretations = {
    'Intercept': 'Baseline predicted charges when all predictors are zero',
    'age': 'Effect of one additional year of age, holding other variables constant',
    'obese': 'Effect of obesity (BMI ≥ 30) among non-smokers',
    'smoker': 'Baseline shift in charges when smoker = 1 vs. 0, among non-obese',
    'obese:smoker': 'Additional effect when both obese and smoker: obese smokers face higher charges',
    'children': 'Effect of one additional dependent, holding other variables constant'
}

ci = m6.conf_int(alpha=0.05)
m6_summary = pd.DataFrame({
    'Coefficient': m6.params.round(3),
    'Std. Error': m6.bse.round(3),
    't-statistic': m6.tvalues.round(3),
    'p-value': m6.pvalues.round(3),
    '95% CI lower': ci[0].round(3),
    '95% CI upper': ci[1].round(3)
})
m6_summary.index = [label_map.get(x, x) for x in m6_summary.index]
m6_summary.index.name = 'Term'
m6_summary['Interpretation'] = [interpretations.get(idx, '') for idx in m6_summary.index]

print('M6 Coefficient Summary')
print('=' * 80)
display(m6_summary)

#### Key Takeaways from M6

Smoking is the strongest predictor of medical insurance charges: the smoker coefficient indicates a large baseline shift in charges for smokers relative to non-smokers. Obesity (BMI ≥ 30) has a modest effect among non-smokers, but its effect is substantially amplified for smokers through the interaction term (obese:smoker). This interaction captures the fact that obese smokers face much higher charges than obese non-smokers or non-obese smokers. Using the binary obese indicator rather than continuous BMI allows the model to capture the clinical threshold at BMI = 30, which better reflects the charge structure in the data and yields superior fit (higher R², lower RMSE) compared to M4.